**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/04-unet-from-paper.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/unet-figure-1) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Cursor](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7D · U-Net from the Paper (Ronneberger et al., 2015)

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| 90 min (training ≈ 1–2 min on CPU) | Google Colab (CPU is enough, GPU is faster) or Cursor / VS Code on your laptop | the exact 2015 U-Net (572 → 388, 31.0 M parameters) and a modern U-Net that segments circles, squares and triangles, plus an ablation that proves skip connections matter |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. 

📄 **Paper:** O. Ronneberger, P. Fischer, T. Brox, *U-Net: Convolutional Networks for Biomedical Image Segmentation*,
MICCAI 2015 · [arXiv:1505.04597](https://arxiv.org/abs/1505.04597) · [PDF](https://arxiv.org/pdf/1505.04597) ·
read **§2 (Network Architecture, ½ page!)** and **Figure 1**.

> *"It consists of a contracting path (left side) and an expansive path (right side). The contracting path … consists of
> the repeated application of two 3x3 convolutions (unpadded convolutions), each followed by a rectified linear unit
> (ReLU) and a 2x2 max pooling operation with stride 2 for downsampling. At each downsampling step we double the number
> of feature channels. Every step in the expansive path consists of an upsampling of the feature map followed by a 2x2
> convolution ("up-convolution") that halves the number of feature channels, a concatenation with the correspondingly
> cropped feature map from the contracting path, and two 3x3 convolutions, each followed by a ReLU. … At the final layer
> a 1x1 convolution is used to map each 64-component feature vector to the desired number of classes. In total the
> network has 23 convolutional layers."* (§2)

**Segmentation** = classify **every pixel** (Lecture 4 showed masks from a pretrained model; today we build one).

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

## 0 · Read Figure 1 like an engineer: fill in the shape table

Open Figure 1 of the paper (or the lecture slide). Write the numbers **before** you open the answer.

| Where | Size | Channels |
|---|---|---|
| input tile | 572 × 572 | 1 |
| after the first two 3 × 3 convs | ? | ? |
| after the first max-pool | ? | ? |
| bottom of the U (after its two convs) | ? | ? |
| output map | ? | ? |

<details><summary><b>Answer</b></summary>

568 × 568 × 64 → 284 × 284 × 64 → 28 × 28 × 1024 → 388 × 388 × 2. Each unpadded 3 × 3 conv removes 2 pixels
(`conv_out(572, 3) = 570`), each pool halves, each up-conv doubles. 572 − 388 = 184 pixels are "lost" at the borders;
that is why the paper predicts large images **tile by tile** with mirrored borders (its Figure 2).
</details>

## 1 · The building block: `DoubleConv` (paper version)

### 🖐 Your turn 1: two **unpadded** 3 × 3 convolutions, each followed by ReLU
<details><summary><b>Hint 1</b></summary>

`nn.Sequential(nn.Conv2d(cin, cout, 3), nn.ReLU(inplace=True), nn.Conv2d(cout, cout, 3), nn.ReLU(inplace=True))`

</details>
<details><summary><b>Hint 2</b></summary>

Unpadded = `padding=0`, which is the default of `nn.Conv2d`.

</details>

In [ ]:
class DoubleConvValid(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = None   # TODO

    def forward(self, x):
        return self.net(x)

In [ ]:
C.check_double_conv(DoubleConvValid)

## 2 · "Copy and crop": `center_crop`

The skip connection carries a 64 × 64 map from the left, but the right side is only 56 × 56 (it lost border pixels).
The paper crops the centre of the left map so that both have the same size before concatenating.

### 🖐 Your turn 2: `center_crop(x, size)` for (B, C, H, W) tensors
<details><summary><b>Hint 1</b></summary>

`top = (H - size) // 2`, `left = (W - size) // 2`

</details>
<details><summary><b>Hint 2</b></summary>

`return x[..., top:top + size, left:left + size]` (the `...` keeps B and C).

</details>

In [ ]:
def center_crop(x, size):
    # TODO
    return x

In [ ]:
C.check_center_crop(center_crop)

## 3 · The full 2015 U-Net

The contracting path (5 `DoubleConvValid`s: 64 → 128 → 256 → 512 → 1024 channels, a max-pool between them) is given.
You write the **expansive path**: for each of the 4 levels on the way up:

1. `x = up(x)`: 2 × 2 up-convolution (`ConvTranspose2d`), doubles the size, halves the channels;
2. crop the matching skip map to `x`'s size and **concatenate along channels** (`dim=1`);
3. `x = conv(x)`: a `DoubleConvValid`.

### 🖐 Your turn 3: the `for` loop in `forward`
<details><summary><b>Hint 1</b></summary>

The skips were stored top-down; on the way up you need them bottom-up: `reversed(skips)`.

</details>
<details><summary><b>Hint 2</b></summary>

`for up, conv, skip in zip(self.ups, self.up_convs, reversed(skips)):`

</details>
<details><summary><b>Hint 3</b></summary>

`x = up(x)`, `x = torch.cat([center_crop(skip, x.shape[-1]), x], dim=1)`, `x = conv(x)`.

</details>

In [ ]:
class UNetPaper(nn.Module):
    def __init__(self, in_ch=1, n_classes=2):
        super().__init__()
        ch = [64, 128, 256, 512, 1024]
        self.downs = nn.ModuleList([DoubleConvValid(in_ch if i == 0 else ch[i - 1], ch[i]) for i in range(5)])
        self.pool = nn.MaxPool2d(2)
        self.ups = nn.ModuleList([nn.ConvTranspose2d(ch[i], ch[i - 1], 2, stride=2) for i in range(4, 0, -1)])
        self.up_convs = nn.ModuleList([DoubleConvValid(ch[i], ch[i - 1]) for i in range(4, 0, -1)])
        self.head = nn.Conv2d(64, n_classes, 1)      # the final 1x1 conv

    def forward(self, x):
        skips = []
        for i, down in enumerate(self.downs):        # contracting path
            x = down(x)
            if i < 4:
                skips.append(x)
                x = self.pool(x)
        # TODO: expansive path (4 levels)
        return self.head(x)

In [ ]:
unet_paper = UNetPaper()
C.check_unet_paper(unet_paper)
C.trace_shapes(unet_paper, torch.zeros(1, 1, 572, 572));   # compare every line with Figure 1 (≈ 3 s on CPU)

**23 convolutional layers?** Count the `Conv2d` + `ConvTranspose2d` rows above: 18 (in the DoubleConvs) + 4 (up-convs) + 1 (head) = 23. ✔ The paper and your code agree.

## 4 · From the 2015 paper to today's U-Net

The 2015 network is big (31 M parameters) and uses tiling. Practically every modern U-Net changes three details; the
idea (encoder, decoder, skip connections) is unchanged:

| 2015 paper | Today | Why |
|---|---|---|
| unpadded convs, crop skips | `padding=1`, no crop | output = input size, no tiling |
| no normalisation | `BatchNorm2d` after each conv | trains faster, less tuning (Ioffe & Szegedy, 2015) |
| 64 … 1024 channels, depth 4 | `base` 16–64, depth 3–5 | fit the data and the GPU |

Below is such a U-Net (given). `use_skips=False` removes the skip connections, for an experiment later.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, n_classes=4, base=16, depth=3, use_skips=True):
        super().__init__()
        self.use_skips = use_skips
        ch = [base * 2 ** i for i in range(depth + 1)]                       # 16, 32, 64, 128
        self.downs = nn.ModuleList([DoubleConv(in_ch if i == 0 else ch[i - 1], ch[i]) for i in range(depth + 1)])
        self.pool = nn.MaxPool2d(2)
        self.ups = nn.ModuleList([nn.ConvTranspose2d(ch[i], ch[i - 1], 2, stride=2) for i in range(depth, 0, -1)])
        self.up_convs = nn.ModuleList([DoubleConv(ch[i - 1] * (2 if use_skips else 1), ch[i - 1]) for i in range(depth, 0, -1)])
        self.head = nn.Conv2d(base, n_classes, 1)

    def forward(self, x):
        skips = []
        for i, down in enumerate(self.downs):
            x = down(x)
            if i < len(self.downs) - 1:
                skips.append(x)
                x = self.pool(x)
        for up, conv, skip in zip(self.ups, self.up_convs, reversed(skips)):
            x = up(x)
            if self.use_skips:
                x = torch.cat([skip, x], dim=1)
            x = conv(x)
        return self.head(x)

C.trace_shapes(UNet(), torch.zeros(2, 1, 64, 64), max_rows=12);

## 5 · Make a dataset (no download needed)

Real medical data needs permission and a GPU, so we generate our own: grey images with **circles (class 1), squares
(class 2) and triangles (class 3)** on a noisy background, with a perfect mask for every pixel. The helper
`C.make_shapes_dataset` can call **your** drawing function for each shape.

### 🖐 Your turn 4: finish `draw_shape` (the square is missing)
<details><summary><b>Hint 1</b></summary>

`cv2.rectangle(image, (x1, y1), (x2, y2), colour, -1)` fills a rectangle (thickness −1).

</details>
<details><summary><b>Hint 2</b></summary>

Draw on `img` with colour `value` AND on `mask` with colour `2`, corners `(cx - r, cy - r)` and `(cx + r, cy + r)`.

</details>

In [ ]:
import cv2

def draw_shape(img, mask, kind, cx, cy, r, value):
    if kind == 1:                                          # circle
        cv2.circle(img, (cx, cy), r, value, -1)
        cv2.circle(mask, (cx, cy), r, 1, -1)
    elif kind == 2:                                        # square
        pass  # TODO: draw the square on img (brightness value) and on mask (label 2)
    else:                                                  # triangle
        pts = np.array([[cx, cy - r], [cx - r, cy + r], [cx + r, cy + r]], np.int32)
        cv2.fillPoly(img, [pts], value)
        cv2.fillPoly(mask, [pts], 3)

In [ ]:
C.check_dataset(draw_shape)
Xtr, Ytr = C.make_shapes_dataset(800, 64, seed=0, draw_fn=draw_shape)     # training set
Xte, Yte = C.make_shapes_dataset(200, 64, seed=1, draw_fn=draw_shape)     # test set: different random shapes
print("images:", tuple(Xtr.shape), Xtr.dtype, "· masks:", tuple(Ytr.shape), Ytr.dtype, "· classes:", Ytr.unique().tolist())
fig, ax = plt.subplots(2, 6, figsize=(12, 4.2))
for i in range(6):
    ax[0, i].imshow(Xtr[i, 0], cmap="gray"); ax[1, i].imshow(Ytr[i], cmap="viridis", vmin=0, vmax=3)
    ax[0, i].axis("off"); ax[1, i].axis("off")
ax[0, 0].set_title("image"); ax[1, 0].set_title("mask: 0 bg · 1 ○ · 2 □ · 3 △", loc="left")
plt.tight_layout(); plt.show()

## 6 · How good is a mask? The Dice score

$$\mathrm{Dice}(A, B) = \frac{2\,|A\cap B|}{|A| + |B|}\in[0, 1]$$
$A$ = pixels predicted as class *c*, $B$ = pixels that really are class *c*. 1 = perfect overlap, 0 = none.
(IoU $= |A\cap B| / |A\cup B|$ is its cousin; Dice = 2·IoU / (1 + IoU).)

### 🖐 Your turn 5: `dice(pred, true, c)`
<details><summary><b>Hint 1</b></summary>

`a = (pred == c)` and `b = (true == c)` are bool masks; `(a & b).sum()` counts the overlap.

</details>
<details><summary><b>Hint 2</b></summary>

If `a.sum() + b.sum() == 0` the class is absent in both: return `torch.tensor(1.0)`.

</details>

In [ ]:
def dice(pred, true, c):
    # TODO
    return torch.tensor(0.0)

In [ ]:
C.check_dice(dice)

## 7 · Train the U-Net (≈ 1 min on CPU)

Loss: pixel-wise cross-entropy, exactly the paper's "pixel-wise soft-max … combined with the cross entropy loss" (§3).
`F.cross_entropy` takes logits (B, 4, H, W) and labels (B, H, W). Before training on everything we do the **#1 sanity
check** for any new implementation: can the model memorise a single batch?

In [ ]:
C.overfit_one_batch(UNet(), Xtr[:8], Ytr[:8], F.cross_entropy, steps=100, lr=3e-3);

In [ ]:
def train_unet(use_skips=True, epochs=10, bs=32, lr=3e-3, verbose=True):
    C.seed_everything(0)
    model = UNet(use_skips=use_skips).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    steps = epochs * math.ceil(len(Xtr) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)   # warm up, then cool down
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            x, y = Xtr[idx].to(device), Ytr[idx].to(device)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if verbose:
            print(f"epoch {ep + 1:2d}/{epochs} · loss {loss.item():.3f} · {time.time() - t0:.0f} s")
    return model

@torch.no_grad()
def evaluate(model):
    model.eval()
    pred = model(Xte.to(device)).argmax(1).cpu()
    scores = [dice(pred, Yte, c).item() for c in (1, 2, 3)]
    return pred, scores

unet = train_unet(use_skips=True)
pred, scores = evaluate(unet)
print("Dice  circle %.3f · square %.3f · triangle %.3f" % tuple(scores))
C.check_unet_dice(sum(scores) / 3)

In [ ]:
fig, ax = plt.subplots(3, 6, figsize=(12, 6.2))
for i in range(6):
    ax[0, i].imshow(Xte[i, 0], cmap="gray"); ax[1, i].imshow(Yte[i], vmin=0, vmax=3); ax[2, i].imshow(pred[i], vmin=0, vmax=3)
    for r in range(3):
        ax[r, i].axis("off")
for r, t in enumerate(["test image", "true mask", "U-Net prediction"]):
    ax[r, 0].set_title(t, loc="left")
plt.tight_layout(); plt.show()

## 8 · An ablation: do the skip connections matter?

An **ablation** removes one part of a model to measure what it contributes. Almost every paper has an ablation table;
now you make one. Same data, same training, only `use_skips=False` (≈ 1 min).

In [ ]:
unet_noskip = train_unet(use_skips=False, verbose=False)
pred_ns, scores_ns = evaluate(unet_noskip)
print(f"{'':12s}{'circle':>8s}{'square':>8s}{'triangle':>9s}{'mean':>8s}")
for name, s in [("with skips", scores), ("no skips", scores_ns)]:
    print(f"{name:12s}" + "".join(f"{v:8.3f}" for v in s) + f"{sum(s) / 3:9.3f}")
gap = (pred == Yte).float().mean((1, 2)) - (pred_ns == Yte).float().mean((1, 2))
k = int(gap.argmax())                                  # the test image where the skips help most
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
for a, im, t in [(ax[0], Xte[k, 0], "image"), (ax[1], Yte[k], "truth"), (ax[2], pred[k], "with skips"), (ax[3], pred_ns[k], "no skips")]:
    a.imshow(im, cmap="gray" if t == "image" else None, vmin=0, vmax=1 if t == "image" else 3); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

**What you should see:** without skips the shapes come out blurry and corners get rounded; triangles and squares are
confused. The decoder only receives the coarse 8 × 8 bottom of the U and has to *guess* the fine detail. The skip
connections hand it the sharp, high-resolution features from the encoder: that is the paper's main idea, now proved
with your own numbers.

## ✍️ Your turn
1. Why does the 2015 U-Net output 388 × 388 for a 572 × 572 input, while our modern U-Net keeps 64 × 64?
2. Make the U-Net **smaller** (`base=8`) and **deeper** (`depth=4`). Record parameters and Dice in a table. Which matters more here?
3. Make the task harder: `noise=0.3` in `make_shapes_dataset`. How much does Dice drop?
4. ⭐ The paper's **weighted loss** (Eq. 2) gives more weight to pixels between touching objects. Use `F.cross_entropy(..., weight=torch.tensor([...]))` to give triangles (the rarest class) a bigger weight. Does the triangle Dice improve?
5. ⭐ Use your U-Net on a real dataset: the [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) masks via `torchvision.datasets.OxfordIIITPet(..., target_types="segmentation")` (needs a GPU runtime).

**Next:** [Lab 7E · ResNet from the paper](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/05-resnet-from-paper.ipynb)